# Visualizing Space Using Maps: An Introduction to Spatial Analysis

<div class="alert alert-info">

Welcome back! This activity is part of an introduction to computational notebooks, designed specifically for K-12 educators.

In this notebook, we'll dive into the beginning of **spatial analysis** by **presenting the usefulness of maps for exploring data.**

</div>

Spatial analysis is used in various fields and disciplines, including political science mapping election polls, public health creating maps on disease prevalence, and civil engineering creating maps for roads and traffic flow.

<hr style="border: 5px solid #003262;" />

## Key Ideas in Spatial Analysis

<br/>

To explore some of the key ideas related to the code, math, and science of spatial analysis, in this notebook we will be exploring **life expectancy**.

**Our Learning Goals:**

* ⏰ **Points, Polygons, and Centroids:** Learn when to use centroids and polygons for visualizations and calculations.

* 📈 **Creating Static and Interactive Maps:** Use `matplotlib.pyplot` and `plotly.express.choropleth` to present data for varying uses of static and interactive plots to see differenes between visualized areas.

* 🌍 **Exploring Potential Reasons for Differences in State Life Expectancies:** 
Explore what life expectancy looks like in different states in the United States and potential causes for these differences.

<hr style="border: 1px solid #fdb515;" />

We'll start with a quick intro into spatial analysis by focusing on: **the components of every map and uses for the different map components.**

---

## Part I: Loading the Data

We are going to load a dataset of state-level data for several public health topics for the United States. 

The data comes from the County Health Rankings and Roadmaps, which gathers data on public health topics from various federal agencies. We will be loading in other data to further explain spatial components in our examples.  

Remember to run code cells like the one below, to load the data and set the variables you need to do your analysis.

Because our data has spatial components, such as polygons, we require a format that keeps the geometric location and attribute information about our geographic features. Here we are using a shapefile, which allows our `geometry` column to be read as multiple coordinates rather than a long string of numbers.

In [ ]:
# %pip install -r requirements.txt

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

df = gpd.read_file('./data/life_expectancy/state_2025_map_data.shp').sort_values(['state', 'state_name'])

# let's see the state abbreviation, state name, and the points that make up each state
df[['state', 'state_name', 'geometry']].head()

We'll use this dataset to practice coding, statistics, and visualizations for spatial analysis.  

Remember: good documentation explains the **why** (in text) and the **how** (in `# code comments`).  

---

## Part II: Exploring Spatial Data Components

Before we start diving into answer questions about life expectancy in the United States, let's first dive into what our spatial components mean.

We quickly mentioned the `geometry` column, which is where we have the polygons for each state. A **Polygon** is the area filled in when connecting lines together to form an outline around some meaningful area. We could think about county limits creating an outline that would determine a polygon. 

To draw polygons on a map, we first need coordinates. **Coordinates** are the building blocks for many spatial analyses. They are comprised of a pair of longitude (x-axis) and latitude (y-axis) values that when combined can be drawn as a dot on a map. These dots, or coordinates, are our data **points**.

Polygons can be created from coordinates that are then connected in a specific order so that the line connecting each point will finish at the starting point. 

Below we have a quick example that shows how the order of the coordinates can influence how points are connected. We are less interested in the code and are more interested in how the order of coordinates influences our polygons.

In [ ]:
example = pd.DataFrame({'x': [2, 2, 4, 4, 2],
                        'y': [2, 4, 4, 2, 2],
                        'order' : [1, 3, 2, 4, 5], # order of how points will be plotted
                        'correct_order': [1, 2, 3, 4, 5]}) # order of how points will be plotted

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

df_incorrect = example.sort_values(by = 'order') # makes the points follow this order
# plotting the points and connecting them with solid lines
ax1.plot(df_incorrect['x'], df_incorrect['y'], marker = 'o', linestyle = '-', color = 'red')
# filling in the area that is contained by the lines
ax1.fill(df_incorrect['x'], df_incorrect['y'], alpha = .3, color = 'red')

ax1.set_title('Polygon in the Incorrect Order')
ax1.grid(True)

df_correct = example.sort_values(by = 'correct_order') # reordering to make the points follow this order
ax2.plot(df_correct['x'], df_correct['y'], marker = 'o', linestyle = '-', color = 'green')
ax2.fill(df_correct['x'], df_correct['y'], alpha = .3, color = 'green')

ax2.set_title('Polygon in the Correct Order')
ax2.grid(True)

plt.tight_layout()
plt.show()

As seen above, when we don't connect the dots in a specific order, we do not get the shape we wanted. The good thing is that when your data is configured in a polygon, the correct order is most likely going to be specified. 

Additionally, when we have a collection of polygon shapes, it is categorized in our data as a multipolygon. Once again, our data will have it listed as a multipolygon and it will draw out the polygons for us. 

In our state-level data, we will see this below with California due to the islands off the coast. 

In [ ]:
#we'll look at California to start with our example
ca_geom = df.loc[4, 'geometry'] # California has 11 polygons

# changing this value from 0-10 can show you each polygon in California
ca_geom.geoms[10]

If we wanted to plot all of our coordinates, we would have a long list of coordinates that correspond to each longitude and latitude combination to draw out the outline of California. For simplicity, we will look at the first five coordinates for the California mainland and then we can plot the state outline afterward.

In [ ]:
ca_geom_list = list(ca_geom.geoms)
mainland_geom = ca_geom_list[10] # mainland polygon
catalina_geom = ca_geom_list[0] # catalina island polygon

# putting it into a list make it easier to show specific points
ca_coord = list(mainland_geom.exterior.coords) 
catalina_coord = list(catalina_geom.exterior.coords)

# this allows us to view the first 5 coordinate pairs for California's mainland
print(ca_coord[:5]) 

As we can see, we have the longitude (x-axis) and latitude (y-axis) pairs that can then be plotted. While the `geometry` column connects these points, we can see the rough outline of California by plotting the points. 

We could also plot the polygon for Catalina Island to see where the island is in relation to the California mainland.

In [ ]:
# coordinates for California outline
cadf = pd.read_csv('./data/life_expectancy/ca_long_lat_points.csv') 
# coordinates for catalina island outline
island = pd.read_csv('./data/life_expectancy/catalina_long_lat_points.csv') 

fig_pt, ax_pt = plt.subplots(figsize = (8, 7))

# plot the points for California mainland
ax_pt.scatter(cadf['longitude'], cadf['latitude'], s = 5, color = 'black')
# plot the points for catalina island
ax_pt.scatter(island['longitude'], island['latitude'], s = 5, color = 'red', zorder = 2)
ax_pt.set_title('Plotting CA Mainland & Catalina Island Points')
plt.show()

As you can see, there are a lot of coordinates that make up a polygon. You might also notice that there are gaps between some of these coordinates. This is where our `geometry` columns fills in these gaps with lines connecting these points in the order that the coordinates are in within the data to draw our California outline.

Points are not very useful for drawing outlines since we could use a polygon to get a better look at our data. Let's see how a polygon looks in comparison to our points map.

In [ ]:
# only california data
ca = df.loc[(df['state'] == 'CA')] 

fig_base, ax_base = plt.subplots(figsize = (8, 7))

# plotting only the geometry column to create polygons
ca.plot(ax = ax_base, color = 'white', edgecolor = 'black')
plt.show()

Points can be useful when we want to draw attention to contextual points within our map. For instance, if we are interested in viewing where every hospital in California is located and we represented the hospitals and the outline as points, our plot would get messy quickly. Instead, we can use a polygon to draw the outline of California like we did above and then use points to represent the hospitals in California. 

Let's load in some data on California hospitals below and see where hospitals are located in California.

In [ ]:
# loading in data for hospitals in California
ca_hosp = pd.read_csv('./data/life_expectancy/california_hospitals.csv', index_col = 0)

ca_hosp.head()

In [ ]:
fig_ca, ax_ca = plt.subplots(figsize = (8, 7))

ca.plot(ax = ax_ca, color = 'white', alpha = .1, edgecolor = 'black')
ax_ca.scatter(ca_hosp['longitude'],
              ca_hosp['latitude'],
              alpha = .5,
              s = 5,
              zorder = 2)
ax_ca.set_title('Map of Hospitals in California')
plt.show()

Now that we have points that show every hospital in California, we can incorporate another important spatial component. **Centroids** are coordinates for the average value of longitude and latitude values. While we could calculate centroids for any set of coordinates, centroids are useful when having a lot of points and needing a general understanding of specific points of interest. Using our points for hospitals in California, we could create centroids to get the average location of hospitals for every county in California.

Since California has several counties, let's focus on the county with the most hospitals and counties with 2 hospitals. We are not going to focus on counties with 1 hospital because the average would be the same as that one point. Instead, we can get the average coordinates from 2 hospitals' longitude and latitude values. 

<div class="alert alert-success">

**Considering the plot above,**



* What county do you think will have the most hospitals?



* Based on the map above, could you name a county with 2 hospitals? If so, what county?




### Incorporating County Outlines & Hospital Counts

Let's now incoporate county polygons and get the counts for the most hospitals and the counties with 2 hospitals.

We could then plot them to show our counties of interest and only show data for hospitals in these specific counties. 

In [ ]:
# counts for the county with the most hospitals and counties with 2 hospitals
ca_hosp_counts = pd.read_csv('./data/life_expectancy/ca_hospital_counts.csv', index_col = 0)
# county polygons
ca_county = gpd.read_file('./data/life_expectancy/ca_county.shp')

ca_hosp_counts

In [ ]:
# county coordinates for our specific counties
most_min_ca_hosp = pd.read_csv('./data/life_expectancy/most_and_minimum_ca_hospitals.csv')

fig_minmax, ax_minmax = plt.subplots(figsize = (8, 7))

# plots the counties for the entirety of California
ca_county.plot(ax = ax_minmax, color = 'white', alpha = .1, edgecolor = 'black')
# puts a darker outline around our specific counties
ca_county.loc[ca_county['county'].isin(ca_hosp_counts['county'])].plot(ax = ax_minmax, color = 'white', alpha = .7, edgecolor = 'black')
# plots the hospitals in our specific counties
ax_minmax.scatter(most_min_ca_hosp['longitude'],
              most_min_ca_hosp['latitude'],
              alpha = .5,
              s = 5,
              zorder = 2)
plt.show()

Now that we know what county has the most hospitals and those that have 2 hospitals, we could calculate the centroids by getting the average for the longitude and latitude values of the hospitals for each county.

To get centroids for each county, we can calculate the average longitude and latitude values for each county. This way, we could then plot each county's centroid within the same plot above.

In [ ]:
# grouping our data to get averages for each county
hosp_centroid = most_min_ca_hosp.groupby('county')

# calculating the averages for latitutde and longitude for each county
hosp_centroid = hosp_centroid[['longitude', 'latitude']].mean()

# rounding the centroid coordinates
hosp_centroid = hosp_centroid.round(2).reset_index()

hosp_centroid

In [ ]:
fig_cent, ax_cent = plt.subplots(figsize = (8, 7))

# creates the outlines for each California county
ca_county.plot(ax = ax_cent, color = 'white', alpha = .1, edgecolor = 'black')
# puts an outline around our specific counties
ca_county.loc[ca_county['county'].isin(ca_hosp_counts['county'])].plot(ax = ax_cent, color = 'white', alpha = .7, edgecolor = 'black') 
# plots the hospital points in our specific counties
ax_cent.scatter(most_min_ca_hosp['longitude'],
              most_min_ca_hosp['latitude'],
              alpha = .25,
              s = 5,
              zorder = 2)
# plots the hospital centroids we created for our specific counties
ax_cent.scatter(hosp_centroid['longitude'],
              hosp_centroid['latitude'],
              alpha = .5,
              s = 5,
              color = 'red',
              zorder = 3)
ax_cent.set_title('Map of Centroids for Los Angeles County with the\nMost Hospitals and Counties with only 2 Hospitals in California')
plt.show()

We can see the centroids' positions in relation to the points for hospitals. For the counties with 2 hospitals, you can see that each centroid is in between the two hospital points showing that the average of the coordinates is between the points.

<div class="alert alert-success">

**Considering the plot above,**


<b>FOR MICHELLE: I'm not sure about the questions under the first bullet point.</b>

* What do you notice about the map?/Are there any issues that you notice from our centroid calculations?



* Why do you think calculating centroids could be useful for hospitals in this example?



* What are some questions you could explore using maps like the USA map above?




### When to use: Points, Centroids, or Polygons

In our hospital example, each spatial component is useful for different questions.

We can use points when we want to see every data point. This could be helpful when we want to see where there are clusters of hospitals and where there are scarce numbers of hospitals.

Centroids could be helpful when there are too many data points to make sense of the clustering of data points. For instance, if we were looking at all hopsitals in the United States, it might be difficult to see trends when there are too many points. Instead, by using centroids, we could see the general center for where hospitals are in each state. 

Lastly, polygons can be useful for when our data applies to the entire area.

Points = Coordinates or 1 pair of longitude and latitude values
Centroids = Average coordinates for several points of interest
Polygons = Area that is determined from an outline of points

<hr style="border: 5px solid #003262;" />

# Expecting More From Your Data: Using Spatial Data to Compare State Life Expectancy Estimates for 2025

Recent statistics came out from the National Center for Health Statistics that have shown the average life expectancy in the United States has recovered from averages during the COVID-19 pandemic at about 79 years old. However, patterns are still present that show public health disparities among states.

To dive into the data, we are first going to examine data visually using traditional methods, followed by showing the usefulness of polygons, and finally getting the most out of our notebook by exploring an interactive map.

<div class="alert alert-success">

**Let's Think Like a Scientist**



* What do you think you will find in the data?



* Is there any state that you think will have a drastically lower average life expectancy than the other states? Why did you choose this state?



* How about a drastically greater average life expectancy? Why did you choose this state?




## Part I: Traditional Plots

Let's use traditional plots to look at life expectancy for states and District of Columbia (DC).

In [ ]:
fig_bar, ax_bar = plt.subplots(figsize = (8, 9))

# plots horizontal bars to more easily read state abbreviations
ax_bar.barh(df['state'], df['life'])
ax_bar.set_title('Average Life Expectancy Across the United States')
ax_bar.set_xlim(0, 100)
ax_bar.set_xlabel('Percentage (%)')
plt.show()


<div class="alert alert-success">

* What are your first thoughts as you look at this data?



* What conclusions can you draw from the plot?



* What might help you see trends better?





We can see slight differences between states, but it is difficult to see with so many different states near one another. Additionally, unless you are an expert at knowing where all 50 states and DC, it is hard to picture if neighboring states are similar.

For traditional plots, we might add additional components to better interpret state differences in life expectancy. One thing we could add is to change the color by region of the United States. Most often, states are broken down into four or five regions. Four regions tends to be the most common, which includes the West, Northeast, Midwest, and the South. 

Below we will try out incorporating region into our plot to helpfully make it more clear on trends in the data.

In [ ]:
# we are assigning specific colors to our regions
region_color = {'South': 'purple', 'West': 'red', 'Midwest': 'green', 'Northeast': 'blue'}
# making sure that our colors will be recognized when using 'region'
region_colors = df['region'].map(region_color)

import matplotlib.patches as mpatches

fig_bar_reg, ax_bar_reg = plt.subplots(figsize = (8, 7))

# we'll add colors for our regions
ax_bar_reg.barh(df['state'], df['life'], facecolor = region_colors)
fig_bar_reg.suptitle('Average Life Expectancy Across the United States')
# we'll also add a legend that correctly assigns color to regions
legend_handles = [mpatches.Patch(color=color, label=region) 
                  for region, color in region_color.items()]
ax_bar_reg.legend(handles = legend_handles, title = 'Regions')
ax_bar_reg.set_xlim(0, 100)
ax_bar_reg.set_title('By Regions')
ax_bar_reg.set_xlabel('Percentage (%)')
plt.show()

Now we can see there appears to be differences based on the region of the United States. However, the way the plot is organized right now makes it difficult to make comparisions between specific states.

This is a good example of a map being used to see differences between states. Static maps are good at visualizing trends when you have prior information, such as knowing where the states are at generally.

In [ ]:
fig_static, ax_static = plt.subplots(figsize = (8, 7))

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# FOR MICHELLE: Should we just keep the original legend
# to reduce cognitive load of extra code

# creates a legend that can be put on the map
cax = inset_axes(
    ax_static,
    width = '50%',      # width of legend
    height = '5%',      # height of legend
    loc = 'upper right', # location of the legend
    borderpad = 1) # allows for whitespace for legend from the plot border

df.plot(ax = ax_static,
        column = 'life', # our life expectancy variable
        cmap = 'YlGnBu', # Colors range from yellow to green to blue
        edgecolor = 'black',
        legend = True,
        cax = cax, # adds our specialized legend
        legend_kwds = {'orientation': 'horizontal'}) # makes sure our legend is correctly oriented
ax_static.set_title('Life Expectancy in the United States')
ax_static.set_xlim([-180, -65]) # makes sure the entire US is included in the map
plt.show()

While this plot is helpful to start seeing how states are grouped together based on average life expectancy, we lose out on one component that was useful in our traditional methods, the actual values for each state.

In order to get the best of both plots, we can introduce another layer for more data exploration through the use of interactive plots.

In [ ]:
state_map = px.choropleth(df,
                    geojson = df['geometry'], # draws our polygons
                    locations = df.index, # makes sure our polygons align with each state correctly
                    color = 'life', # life expectancy
                    color_continuous_scale = px.colors.sequential.YlGnBu, # same color palette
                    hover_data = ['state_name', 'life'], # what data is shown when hovering over a polygon
                    title = 'Life Expectancy in the United States (2025)',
                    scope = 'usa')
state_map.show()


📈 **To assist with navigating the interactive plot, below are some friendly shortcuts that could help.**

* Zooming in and out can be accomplished using two fingers on a trackpad (similar to scrolling up and down a page)

* With the default settings, clicking and holding on the map and using another finger to move around the map allows you to pan around the map

* If you find yourself zooming in/out too much, double clicking on the map will reset it to the default position.

* If you are trying to scroll past the map and find yourself accidentally zooming in/out of the map, you can move your mouse cursor to the left or right of the general area of the map, and you should be able to move past the interactive plot. 

* There are also additional ways of exploring the map, which are located on the top right above the legend.

<div class="alert alert-success">

**Explore the interactive plot above**


* What does this provide that the static plot did not?



* What life expectancy suprises you? 




* What patterns can we now see from our maps in comparison to the more traditional plots shown before?



</div>

### Let's explore what could be causes for state differences in average life expectancy.

Below are some other variables in the data that could help in explaining the state differences in life expectancy.

| Variable | Description |
|---|---|
| statecode | State FIPS code |
| countycode | County FIPS code |
| fipscode | Combined FIPS code |
| state | State abbreviation |
| county | County name |
| year | Year of data |
| region | Geographic region of the United States |
| state_name | Full name of the state |
| flu_vaccin | Percentage of each state that received an annual flu shot |
| flu_aian | Percentage of each state's American Indian/Alaskan Native population that received an annual flu shot |
| flu_asian | Percentage of each state's Asian population that received an annual flu shot |
| flu_black | Percentage of each state's Black/African American population that received an annual flu shot |
| flu_hisp | Percentage of each state's Hispanic population that received an annual flu shot |
| flu_white | Percentage of each state's White population that received an annual flu shot |
| exercise | Percentage of each state's population that has access to exercise opportunities |
| prev_hosp | Number of preventable hospital stays in each state |
| air_poll | Average air pollution from particulate matter in each state |
| water_vio | Number of drinking water violations in each state |
| income_in | Ratio of household income at the 80th percentile to income at the 20th percentile in each state |
| life | Average number of years a person can expect to live |
| life_aian | Average number of years a American_indidan/Alaskan Native person can expect to live |
| life_asian | Average number of years a Asian person can expect to live |
| life_black | Average number of years a Black/African American person can expect to live |
| life_hisp | Average number of years a Hispanic person can expect to live |
| life_white | Average number of years a White person can expect to live |
| life_nhopi | Average number of years a Native Hawaiian/Other Pacific Islander person can expect to live |
| diabetes | Percentage of adults diagnosed with diabetes in each state |
| hiv | Number of people aged 13 years and older living with a diagnosis of human immunodeficiency virus (HIV) infection per 100,000 population |
| obesity | Percentage of adults with a BMI of 30 or more |
| limit_food | percentage of low-income residents living far from a grocery store (>10 miles in rural areas, >1 mile in non-rural areas) |
| food_in | Percentage of the population who did not have access to a reliable source of food |
| lack_sleep | Percentage of adults who report fewer than 7 hours of sleep on average |
| alcveh_fat | Percentage of driving deaths with alcohol involvement |
| drug_dose | Number of drug poisoning deaths per 100,000 population |
| smoking | Percentage of adults who are current smokers |
| inactivity | Percentage of adults who report no leisure-time physical activity |
| prk_access | Percentage of the population living within a half mile of a park |
| med_income | Median household income |
| rural | Percentage of the population living in a rural area |

Below is our first attempt at trying to see what can be related to state differences in life expectancy.

First, we'll get the descriptive statistics of percentage of each state that received an annual flu vaccine (`flu_vaccin`), along with our life expectancy variable to see if there is any patterns we can see.

Then we can plot our flu vaccine data to see if that shows us anything of interest about life expectancy. By having the flu vaccine (`flu_vaccin`) percentages as polygon colors, we can only see on the map one variable of interest. However, we have already added life expectancy (`life`) in our hover data so when we hover over a specific state we can now see the average flu vaccine percentage and average life expectancy for each state.

If this doesn't tell us enough then we could add more variables or other variables to get descriptive statistics for and then change the color of our map to see if there are any patterns to uncover similar to the map on life expectancy alone.

In [ ]:
# explore the data a little more
# include whatever variable you would like to know more about
var_stat = ['life', 'flu_vaccin']

# we'll look at descriptive statistics of each variable of interest
df[var_stat].describe()

In [ ]:
explore_map = px.choropleth(df,
              geojson = df['geometry'],
              locations = df.index,
              color = 'smoking', # change the variable name to help make conclusions
              color_continuous_scale = px.colors.sequential.YlGnBu,
              hover_data = ['state_name', 'life'], # include data to view when hovering over states for other variables
              # this will automatically include whatever variable is listed under color
              title = 'Potential Causes for Differences in Life Expectancy in the United States (2025)',
              scope = 'usa')
explore_map.show()

<div class="alert alert-success">

**Variable Selection**


* Why did you choose the variable(s) you chose?


    
* From the variable(s) you chose, what conclusions did you draw?
    
    
* What did you find surprising?


</div>

<details>
<summary>Let's dive into some choices you could have made. (Click on this text to show some choices.)</summary>

If you found yourself looking at the map of several variables, then you are on the right track for thinking about public health issues. There is rarely one cause for any public health outcome. For life expectancy, you may have saw that there are discrepancies between racial/ethnic groups. You might have seen that the map looks similar when including other epidemic health outcomes, like obesity. You may have focused on adverse health behaviors that could reduce one's life expectancy, like smoking or inactivity. Finally, you could have seen state inequities from national systematic issues like not having places to exercise, food insecurity, median household income, or how much of the state is rural.
</details>

Examining life expectancy shows the complexity of answer public health issues. Below are some links to other resources that reflect more in-depth exploration of data on life expectancy.

* [The Equality of Opportunity Project brief](http://www.equality-of-opportunity.org/health/)

* [National Center for Health Statistics brief on mortality in the US](https://www.cdc.gov/nchs/products/databriefs/db548.htm)

* [Article on US Life Expectancy](https://www.scientificamerican.com/article/u-s-life-expectancy-hits-all-time-high/)

<hr style="border: 1px solid #fdb515;" />

## Curriculum Connections

This notebook shows how using maps can be useful for exploring public health issues at a macro level. This does not always have to be the case, as the scope of interest may be different based on the public health issue. We might want to dive into county level within each state to see how much each county differs within each state on life expectancy to draw additional conclusions from.

For instance, we may be interested in motor vehicle fatalities within a city. Rather than viewing data at a macro level, we may want to look at specific intersections where these fatalities are more likely to occur. Exploring data at a more granular level allows for more specific intervention. For any level of analysis, the fundamentals are the same, where we plot a base map and can add points on top of the map to be able to draw conclusions from.


**HS-EPHS2-3** Use models (e.g., mathematical models, and figures) that are based on empirical evidence identify patterns of health and disease to characterize a public health problem. to Emphasis is on identification of patterns using basic mathematical models (e.g., frequency measures) and figures (e.g., epi curves, graphs, maps) to describe health-related occurrences by person, place, and time. 

**HS-EPHS2-4** Use patterns in empirical evidence to hypotheses. formulate Emphasis is on using descriptive epidemiology, what is known concerning health-related phenomena, and what others have postulated in historical or current contexts to develop testable hypotheses.

--- 

## Credits


